In [1]:
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms, datasets
from torchinfo import summary
from torch.utils.data import DataLoader, Subset

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else: 
    device = "cpu"

In [3]:
device

'mps'

In [4]:
# image size is 224x224 as in the original Swin Transformer paper.

IMG_SIZE = 224
random_seed = 42

In [5]:
# Caltech101 contains some grayscale data; convert everything to RGB for channel consistency

def to_rgb(img):
    return img.convert("RGB")

In [7]:
# data augmentation

train_tf = transforms.Compose([
    transforms.Lambda(to_rgb),
    transforms.RandomAdjustSharpness(sharpness_factor = 2 ,p = 0.1),
    transforms.RandomResizedCrop(IMG_SIZE, scale = (0.6,1), ratio = (0.75,1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), #imagenet standart values
])



In [8]:
test_tf = transforms.Compose([
    transforms.Lambda(to_rgb),
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [10]:
from torch.utils.data import random_split

root = "../data"
test_ratio = 0.2

ds_train_full = datasets.Caltech101(root = root, transform = train_tf, download = True)
ds_test_full = datasets.Caltech101(root = root, transform = test_tf, download = True)

In [19]:
n_total = len(ds_train_full)
n_test = int(n_total * test_ratio)
n_train = n_total - n_test

In [20]:
# train test split

g = torch.Generator().manual_seed(random_seed)
perm = torch.randperm(n_total, generator = g).tolist()

train_idx = perm[:n_train]
test_idx = perm[n_train:]

In [21]:
len(train_idx), len(test_idx)

(6942, 1735)

In [22]:
train_ds = Subset(ds_train_full, train_idx)
test_ds = Subset(ds_test_full, test_idx)

In [23]:
len(train_ds), len(test_ds)

(6942, 1735)

In [24]:
# create the dataloaders

BATCH_SIZE = 32 # original paper uses 1024, but reduced to 32 due to hardware constraints
train_dataloader = DataLoader(train_ds, batch_size = BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_ds, batch_size = BATCH_SIZE, shuffle=False)

In [25]:
# verify dataloader outputs and inspect a random sample

img, label = ds_train_full[1001]
print(label)
print(ds_train_full.categories[label])

2
Leopards


In [26]:
height = IMG_SIZE
width = IMG_SIZE
color_channels = 3
patch_size = 4 # patch size value in the original paper
linear_layer_embed_dim = 96 # C = 96 for Swin-T
number_of_patches = int((height * width) / (patch_size **2)) # referred as "number of tokens" 


linear_embedding_layer_input_shape = (height, width, color_channels)
linear_embedding_layer_output_shape = (number_of_patches, linear_layer_embed_dim)

In [27]:
# expected 

print(linear_embedding_layer_input_shape)
print(linear_embedding_layer_output_shape)

(224, 224, 3)
(3136, 96)
